# Naive Bayes from Scratch.

---

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
data = pd.read_csv("spam_messages.csv")

In [3]:
data.drop(columns=['MessageID', 'CharacterCount', 'WordCount', 'HasURL', 'HasNumber', 'UppercaseRatio'], inplace=True)

## Feature Engineering:

In [4]:
def clean_text(feature):

    feature = feature.str.lower().str.strip().str.replace(r'https?://\S+|www\.\S+', 'URL', regex=True)
    feature = feature.str.replace(r'\b\d{7,15}\b', 'Phone-Number', regex=True)

    return feature

In [5]:
data['Message'] = clean_text(data['Message'])

In [6]:
data.dropna(inplace=True)

In [7]:
data.drop_duplicates(subset='Message', keep='first', inplace=True)

## Train-Test-Split:

In [8]:
X = data['Message']
y = data['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [9]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [10]:
vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

## Building Model:

In [11]:
class MultinomialNB:

    def __init__(self):
        self.class_prob = None
        self.word_prob = None

    def fit(self, X_train, y_train):

        self.X_train = X_train
        self.y_train = y_train

        # P(class):        
        spam = (self.y_train == 1).sum()
        p_spam = spam / self.X_train.shape[0]

        ham = (y_train == 0).sum()
        p_ham = ham / self.X_train.shape[0]

        self.class_prob = np.array([p_ham, p_spam])

        # P(word | class) using laplace smoothing:
        alpha = 1
        V = self.X_train.shape[1]
        
        spam_class = self.X_train[y_train == 1]
        word_spam_count = np.asarray(spam_class.sum(axis=0)).ravel()
        total_spam_words = word_spam_count.sum()
        p_word_spam = (word_spam_count + alpha) / (total_spam_words + alpha*V) #laplace smoothing

        ham_class = self.X_train[y_train == 0] 
        word_ham_count = np.asarray(ham_class.sum(axis=0)).ravel()
        total_ham_words = word_ham_count.sum()
        p_word_ham = (word_ham_count + alpha) / (total_ham_words + alpha*V) # laplace smoothing

        self.word_prob = np.array([p_word_ham, p_word_spam])

    def predict(self, X_test):

        log_spam = np.log(self.class_prob[1]) + (X_test @ np.log(self.word_prob[1]))
        log_ham = np.log(self.class_prob[0]) + (X_test @ np.log(self.word_prob[0]))

        return (log_spam > log_ham).astype(int)

In [12]:
nb = MultinomialNB()

start_time = time.time()
nb.fit(X_train, y_train)
end_time = time.time() - start_time

print("Model Trained Successfully!")
print(f"Time Taken for Training: {end_time:.2f}s")

Model Trained Successfully!
Time Taken for Training: 0.01s


## Result:

In [13]:
y_pred = nb.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred)*100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

print("\n")
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual Spam', 'Actual Ham'], columns=['Predicted Spam', 'Predicted Ham'])
matrix_df

Accuracy: 100.00%

Precision: 100.00%

Recall: 100.00%

F1 Score: 100.00%




,Predicted Spam,Predicted Ham
Actual Spam,19,0
Actual Ham,0,34
